In [1]:
import numpy as np
import random
import time
import os
import re
from collections import defaultdict
import matplotlib.pyplot as plt

In [2]:
def readVectorsSeq(filename):
    P = []
    with open(filename, 'r') as f:
        for line in f:
            values = list(map(float, line.strip().split(',')))
            P.append(np.array(values))
    return P

P = readVectorsSeq("spambase.data")

print("Total points:", len(P))
print("Dimensions:", len(P[0]))

Total points: 4601
Dimensions: 58


In [3]:
def squared_distance(a, b):
    return np.sum((a - b) ** 2)

In [4]:
def kcenter(P, k):
    centers = []
    centers.append(random.choice(P))

    for _ in range(1, k):
        max_dist = -1
        next_center = None

        for p in P:
            min_dist = min(squared_distance(p, c) for c in centers)

            if min_dist > max_dist:
                max_dist = min_dist
                next_center = p

        centers.append(next_center)

    return centers

In [5]:
def kmeansPP(P, k):
    centers = []
    centers.append(random.choice(P))

    for _ in range(1, k):
        distances = []

        for p in P:
            min_dist = min(squared_distance(p, c) for c in centers)
            distances.append(min_dist)

        total = sum(distances)
        probs = [d / total for d in distances]

        next_center = P[np.random.choice(len(P), p=probs)]
        centers.append(next_center)

    return centers

In [6]:
def kmeansObj(P, C):
    total = 0

    for p in P:
        min_dist = min(squared_distance(p, c) for c in C)
        total += min_dist

    return total / len(P)

In [7]:
k = 10
k1 = 20

# k-center timing
start = time.time()
C1 = kcenter(P, k)
end = time.time()
time_taken = end - start

# kmeans++
C2 = kmeansPP(P, k)
obj2 = kmeansObj(P, C2)

# Hybrid
X = kcenter(P, k1)
C3 = kmeansPP(X, k)
obj3 = kmeansObj(P, C3)

print(f"k = {k}, k1 = {k1}")
print(f"k-center time: {time_taken:.4f} sec")
print(f"kmeans++ objective: {obj2:.4f}")
print(f"Hybrid objective: {obj3:.4f}")

k = 10, k1 = 20
k-center time: 3.5699 sec
kmeans++ objective: 36958.2474
Hybrid objective: 47149.7581


In [ ]:
k_values = [5, 10, 15, 20]
kmeans_results = []
hybrid_results = []

runs = 5 

for k in k_values:
    k1 = k * 2
    
    kmeans_avg = 0
    hybrid_avg = 0
    
    for _ in range(runs):
        C2 = kmeansPP(P, k)
        kmeans_avg += kmeansObj(P, C2)

       
        X = kcenter(P, k1)
        C3 = kmeansPP(X, k)
        hybrid_avg += kmeansObj(P, C3)

    kmeans_results.append(kmeans_avg / runs)
    hybrid_results.append(hybrid_avg / runs)

In [ ]:
plt.figure()
plt.plot(k_values, kmeans_results, marker='o', label="kmeans++")
plt.plot(k_values, hybrid_results, marker='s', label="Hybrid")

plt.xlabel("k (Number of clusters)")
plt.ylabel("Objective Value")
plt.title("Clustering Performance Comparison")
plt.legend()
plt.grid()

plt.show()

In [ ]:
print("Observations:")
print("- Hybrid approach often gives better (lower) objective values.")
print("- Increasing k reduces objective value.")
print("- kmeans++ performs better than naive random selection.")

In [ ]:
#part 2 


In [ ]:
stop_words = {
    "a", "an", "the", "they", "these", "this", "for", "is",
    "are", "was", "of", "or", "and", "does", "will", "whose"
}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    
    return words

In [ ]:
def normalize_word(word):
    word = word.lower()
    word = re.sub(r'[^\w\s]', '', word) 
    return word

In [ ]:
inverted_index = defaultdict(list)
page_words = {} 

def add_page(filename):
    with open(filename, 'r') as f:
        text = f.read()
    
    words = preprocess(text)
    page_words[filename] = words
    
    for idx, word in enumerate(words):
        inverted_index[word].append((filename, idx + 1))

In [ ]:
def find_pages(word_input):
    word = normalize_word(word_input)
    
    if word not in inverted_index:
        print(f"No webpage contains word {word_input}")
        return
    
    pages = set(p for p, _ in inverted_index[word])
    print(", ".join(sorted(pages)))

In [ ]:
def find_positions(word_input, page):
    word = normalize_word(word_input)
    
    if page not in page_words:
        print(f"No webpage {page} found")
        return
    
    positions = [pos for p, pos in inverted_index[word] if p == page]
    
    if not positions:
        print(f"Webpage {page} does not contain word {word_input}")
    else:
        print(", ".join(map(str, positions)))

In [ ]:
def process_actions(action_file):
    with open(action_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            
            if parts[0] == "addPage":
                add_page(parts[1])
            
            elif parts[0] == "queryFindPagesWhichContainWord":
                find_pages(parts[1])
            
            elif parts[0] == "queryFindPositionsOfWordInAPage":
                find_positions(parts[1], parts[2])

In [ ]:
process_actions("actions.txt")

In [ ]:
#part 3

In [ ]:
def load_graph(filename):
    edges = set()
    
    with open(filename, 'r') as f:
        for line in f:
            u, v = map(int, line.strip().split())
            edges.add((u, v)) 
    
    return list(edges)

edges = load_graph("whole.txt")

In [ ]:
def build_graph(edges):
    out_links = defaultdict(list)
    nodes = set()
    
    for u, v in edges:
        out_links[u].append(v)
        nodes.add(u)
        nodes.add(v)
    
    return out_links, list(nodes)

out_links, nodes = build_graph(edges)
N = len(nodes)

print("Total nodes:", N)

In [ ]:
beta = 0.8
iterations = 40

r = {node: 1/N for node in nodes}

In [ ]:
for _ in range(iterations):
    new_r = {node: (1 - beta)/N for node in nodes}
    
    for u in nodes:
        if u in out_links:
            share = r[u] / len(out_links[u])
            for v in out_links[u]:
                new_r[v] += beta * share
    
    r = new_r

In [ ]:
sorted_nodes = sorted(r.items(), key=lambda x: x[1], reverse=True)

print("Top 5 nodes:")
for node, score in sorted_nodes[:5]:
    print(node, score)

print("\nBottom 5 nodes:")
for node, score in sorted_nodes[-5:]:
    print(node, score)

In [ ]:
print("Sum of all PageRank values:", sum(r.values()))